# 01 — Intro to Snowpark ML

This notebook walks the whole loop once, end to end, on **synthetic data**:

1. Connect to Snowflake from your local IDE.
2. Load a small mock table and explore it with a **Snowpark DataFrame**.
3. Preprocess and train a model **inside Snowflake** with `snowflake.ml.modeling`.
4. Evaluate it in Snowflake.
5. Register it in the **Model Registry** (schema `HAT_DATASCIENCE_DB.ML_DEV`).
6. Score with the registered model, from Python and from plain SQL.

Later notebooks go deeper on each step. Here the goal is the mental model.

## The mental model

| Piece | What it is | Where it runs |
|---|---|---|
| `Session` | Your authenticated connection: role, warehouse, database, schema | Local |
| Snowpark `DataFrame` | A *lazy description of a query*. Methods like `.filter()` build SQL; nothing runs until an *action* such as `.show()`, `.count()`, `.collect()` or `.to_pandas()` | The SQL runs on the **warehouse** |
| `snowflake.ml.modeling` | scikit-learn / XGBoost-style preprocessors and estimators that take Snowpark DataFrames | `fit()` runs on the **warehouse**, in a temporary stored procedure |
| Model Registry | Models stored as schema objects (`MODEL`), with versions, metrics and callable functions | Inference runs on the **warehouse** |

The big shift from our current pandas workflow: **data stays in Snowflake.** Your laptop sends instructions and
gets back small results. It never holds the full table.

> **Data rule:** demos use only `MOCK_`-prefixed synthetic tables. Never point these notebooks at PHI or
> encounter data.

## 1. Connect

`sfconn.get_session()` reads the `CHARTIS-ANALYTICS` connection from `~/.snowflake/connections.toml`. That
connection uses Okta SSO (`externalbrowser`), so the first run opens a browser tab to sign in. `keyring` caches the
token, so later sessions are usually silent.

We set the context explicitly rather than relying on user defaults, so the notebook behaves the same for everyone.

In [1]:
%load_ext autoreload
%autoreload 2

import sfconn

ROLE = "HAT_DATASCIENCE_ANALYST"
WAREHOUSE = "HAT_DATASCIENCE_WH"  # standard X-Small
DATABASE = "HAT_DATASCIENCE_DB"
SCHEMA = "ML_DEV"  # models developed in this repo live here

session = sfconn.get_session(role=ROLE, warehouse=WAREHOUSE, database=DATABASE, schema=SCHEMA)

('Active session: role="HAT_DATASCIENCE_ANALYST", '
 'warehouse="HAT_DATASCIENCE_WH", database="HAT_DATASCIENCE_DB", '
 'schema="ML_DEV"')


## 2. Create a mock table

In real work the data is already in Snowflake. Here we generate a small synthetic binary-classification dataset
locally and upload it with `write_pandas`, which is fine for demo-sized data. Column names are UPPERCASE because
Snowflake identifiers are case-insensitive *unless quoted*. Uppercase avoids having to write `"quoted"` names later.

In [2]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

FEATURES = [f"F{i}" for i in range(8)]
TABLE = "MOCK_INTRO_CLASSIFICATION"

x, y = make_classification(n_samples=5_000, n_features=len(FEATURES), n_informative=5, random_state=42)
mock = pd.DataFrame(x, columns=FEATURES)
mock.insert(0, "ROW_ID", np.arange(len(mock)))
mock["LABEL"] = y

session.write_pandas(mock, TABLE, auto_create_table=True, overwrite=True, table_type="")
print(f"Wrote {len(mock):,} rows to {DATABASE}.{SCHEMA}.{TABLE}")

Wrote 5,000 rows to HAT_DATASCIENCE_DB.ML_DEV.MOCK_INTRO_CLASSIFICATION


In [3]:
from snowflake.snowpark import functions as F

df = session.table(TABLE)

Calling an **action** sends that SQL to the warehouse and brings back only the (tiny) result:

## 3. Train/test split

`DataFrame.random_split()` exists, but its splits can vary between runs. A deterministic rule on a key column is
reproducible and easy to explain. Here 20% of rows go to the test set.

In [4]:
train_df = df.filter(F.col("ROW_ID") % 5 != 0)
test_df = df.filter(F.col("ROW_ID") % 5 == 0)
print("train:", train_df.count(), " test:", test_df.count())

train: 4000  test: 1000


## 4. Preprocess and train in Snowflake

`snowflake.ml.modeling` mirrors scikit-learn, with one difference: you name the columns up front.

- `input_cols`: feature columns to read
- `label_cols`: target column(s), for estimators only
- `output_cols`: column(s) the step writes

`fit()` is pushed down: Snowflake runs the training inside a temporary stored procedure on the warehouse, next to
the data. Expect a minute or two on X-Small, mostly environment start-up.

The scaler isn't needed for a tree model. It's here to show how preprocessing steps chain in a `Pipeline`.

**Fit on features + label only.** The registered model's input signature comes from the columns of the DataFrame
you `fit()` on. If `ROW_ID` were present, it would become a *required argument* of every prediction function,
including from SQL. Selecting the columns explicitly keeps IDs out of the model. Predicting on a frame that has
extra columns is fine: they pass through untouched.

In [5]:
from snowflake.ml.modeling.pipeline import Pipeline
from snowflake.ml.modeling.preprocessing import StandardScaler
from snowflake.ml.modeling.xgboost import XGBClassifier

SCALED = [f"{c}_SCALED" for c in FEATURES]

pipeline = Pipeline(
    steps=[
        ("scale", StandardScaler(input_cols=FEATURES, output_cols=SCALED)),
        (
            "model",
            XGBClassifier(
                input_cols=SCALED,
                label_cols=["LABEL"],
                output_cols=["PREDICTED_LABEL"],
                n_estimators=200,
                max_depth=4,
                learning_rate=0.1,
                random_state=0,
            ),
        ),
    ]
)
pipeline.fit(train_df.select(*FEATURES, "LABEL"))

c:\Users\bakrefens\repos\snowflake_ml_demos\.venv\Lib\site-packages\snowflake\ml\model\model_signature.py:74: UserWarning: The sample input has 4000 rows. Using the first 100 rows to define the inputs and outputs of the model and the data types of each. Use `signatures` parameter to specify model inputs and outputs manually if the automatic inference is not correct.
  warnings.warn(
c:\Users\bakrefens\repos\snowflake_ml_demos\.venv\Lib\site-packages\snowflake\ml\model\model_signature.py:74: UserWarning: The sample input has 4000 rows. Using the first 100 rows to define the inputs and outputs of the model and the data types of each. Use `signatures` parameter to specify model inputs and outputs manually if the automatic inference is not correct.
  warnings.warn(
c:\Users\bakrefens\repos\snowflake_ml_demos\.venv\Lib\site-packages\snowflake\ml\model\model_signature.py:74: UserWarning: The sample input has 4000 rows. Using the first 100 rows to define the inputs and outputs of the model an

## 6. Evaluate in Snowflake

`predict()` / `predict_proba()` return **Snowpark DataFrames**: the input columns plus new prediction columns.
`snowflake.ml.modeling.metrics` computes the metrics in the warehouse too, so the predictions never leave
Snowflake.

In [6]:
from snowflake.ml.modeling import metrics

scored = pipeline.predict_proba(pipeline.predict(test_df))
proba_cols = [c for c in scored.columns if "PROBA" in c.upper()]
print("probability columns:", proba_cols)

eval_metrics = {
    "test_accuracy": metrics.accuracy_score(
        df=scored, y_true_col_names="LABEL", y_pred_col_names="PREDICTED_LABEL"
    ),
    "test_roc_auc": metrics.roc_auc_score(
        df=scored, y_true_col_names="LABEL", y_score_col_names=proba_cols[-1]
    ),
}
eval_metrics

probability columns: ['PREDICT_PROBA_0', 'PREDICT_PROBA_1']


{'test_accuracy': 0.933, 'test_roc_auc': 0.9774737473747375}

## 7. Register the model in `ML_DEV`

The Registry stores the fitted pipeline as a `MODEL` object in our schema. Each `log_model` call adds a
**version**.\
Two habits worth adopting:

- **Always set `version_name`.** Otherwise Snowflake generates a random one like `FRIENDLY_ROBIN_2`.
- **Pass `metrics=`** so the numbers show on the model version in Snowsight. Metrics are not copied over from
  anywhere automatically.

Registering the same name and version twice is an error, so this cell reuses the version if it already exists.
Bump `VERSION` when you change the model.

In [7]:
from snowflake.ml.registry import Registry

MODEL_NAME = "INTRO_XGB_CLASSIFIER"
VERSION = "V1"

registry = Registry(session=session, database_name=DATABASE, schema_name=SCHEMA)

existing = {m.name for m in registry.models()}
if MODEL_NAME in existing and VERSION in {v.version_name for v in registry.get_model(MODEL_NAME).versions()}:
    mv = registry.get_model(MODEL_NAME).version(VERSION)
    print(f"{MODEL_NAME}/{VERSION} already registered; reusing it")
else:
    mv = registry.log_model(
        pipeline,
        model_name=MODEL_NAME,
        version_name=VERSION,
        metrics=eval_metrics,
        comment="01_intro demo: StandardScaler + XGBClassifier on MOCK_INTRO_CLASSIFICATION",
        target_platforms=["WAREHOUSE"],
    )
mv.show_metrics()

INTRO_XGB_CLASSIFIER/V1 already registered; reusing it


{'test_accuracy': 0.933, 'test_roc_auc': 0.9774737473747375}

A registered version exposes **functions** (the methods you can call). Each takes the feature columns and
returns predictions.\
These "functions" aren't methods of the `mv` object in python. They are registered functions associated with the registered model on Snowflake. To run them, we must call `mv.run(..., function={function_string})`

In [9]:
[f["name"] for f in mv.show_functions()]

['PREDICT', 'PREDICT_PROBA']

## 8. Score with the registered model

### From Python
`mv.run()` takes a Snowpark DataFrame and returns one with predictions appended. Scoring runs in the warehouse.

In [11]:
predictions = mv.run(test_df, function_name="predict_proba")
predictions.select("ROW_ID", "LABEL", *[c for c in predictions.columns if "PROBA" in c.upper()]).show(5)

--------------------------------------------------------------------
|"ROW_ID"  |"LABEL"  |"PREDICT_PROBA_0"      |"PREDICT_PROBA_1"    |
--------------------------------------------------------------------
|0         |1        |0.0037836432456970215  |0.996216356754303    |
|5         |0        |0.9661836624145508     |0.03381630778312683  |
|10        |0        |0.9369118809700012     |0.06308813393115997  |
|15        |1        |0.06885033845901489    |0.9311496615409851   |
|20        |1        |0.015235602855682373   |0.9847643971443176   |
--------------------------------------------------------------------



### From SQL
The same model can be called from any SQL: a worksheet, a view, a scheduled task, or a BI tool. This is what makes
"deployment" mean something different from pulling data into pandas: anyone with access to the model can score in
place.

In [12]:
sql = f"""
WITH m AS MODEL {DATABASE}.{SCHEMA}.{MODEL_NAME} VERSION {VERSION}
SELECT ROW_ID
    , LABEL
    , m!PREDICT({', '.join(FEATURES)}) AS PREDICTION
FROM {TABLE}
WHERE ROW_ID % 5 = 0
LIMIT 5
"""
print(sql)
session.sql(sql).show()


WITH m AS MODEL HAT_DATASCIENCE_DB.ML_DEV.INTRO_XGB_CLASSIFIER VERSION V1
SELECT ROW_ID
    , LABEL
    , m!PREDICT(F0, F1, F2, F3, F4, F5, F6, F7) AS PREDICTION
FROM MOCK_INTRO_CLASSIFICATION
WHERE ROW_ID % 5 = 0
LIMIT 5

---------------------------------------------------------------
|"ROW_ID"  |"LABEL"  |"PREDICTION"                            |
---------------------------------------------------------------
|0         |1        |{                                       |
|          |         |  "F0": 8.003829000302234e-01,          |
|          |         |  "F0_SCALED": 1.819754625819227e-01,   |
|          |         |  "F1": 4.314080274598064e+00,          |
|          |         |  "F1_SCALED": 3.123101723028765e+00,   |
|          |         |  "F2": 3.374232169357195e-01,          |
|          |         |  "F2_SCALED": 1.747854288205891e-01,   |
|          |         |  "F3": -1.128832442937119e+00,         |
|          |         |  "F3_SCALED": -6.575344597839157e-01,  |
|       

The output is an `OBJECT` (a JSON-like map), because a model function can return several columns.

## 9. Where to find this in Snowsight

- **The model:** open the Models page in Snowsight and look under `HAT_DATASCIENCE_DB.ML_DEV`. `INTRO_XGB_CLASSIFIER`
  shows version `V1`, its metrics and its functions.
- **The table:** under Data, `MOCK_INTRO_CLASSIFICATION` is in the same schema.
- **The training job:** query history shows the stored procedure Snowflake ran for `fit()`.

## 10. Clean up

Close the session when you're done. The cell below drops the demo table and model; it's left commented so a
casual "Run All" doesn't delete things.

In [14]:
# session.sql(f"DROP TABLE IF EXISTS {TABLE}").collect()
# registry.delete_model(MODEL_NAME)
session.close()

## Recap

- `snowflake.ml.modeling` trains **next to the data** with a scikit-learn-like API plus `input_cols` /
  `label_cols` / `output_cols`.
- The **Model Registry** turns a fitted model into a versioned schema object you can call from Python *or* SQL.

**Gotchas to remember** (details in `findings.md`):
- Pin package versions to what Snowflake's Anaconda channel offers for our Python (3.11). For example,
  `snowflake-ml-python` 2.7.0 needs `pandas<3`.
- Custom Python code used during `fit()` (a custom loss, say) must be shipped explicitly, e.g. with
  `cloudpickle.register_pickle_by_value(module)`.
- A pipeline of *only* transformers can't be registered. It needs a final estimator.
- The model's input signature is every non-label column of the `fit()` DataFrame. Keep IDs out of it.

**Next:** 02 covers Experiment Tracking: logging and comparing runs.